In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

In [2]:
# Experiments appeared in the article
full_exps = [{"data": 'age', 'loss': 'contrastive', 'alg': 'catboost', 'path': "topo_metrics/csv_results/age_pred_output_2025-10-10 14:37:35.506749_ContrastiveLoss_gru_catboost"},
           {"data": 'age', 'loss': 'contrastive', 'alg': 'mlp', 'path': "topo_metrics/csv_results/age_pred_output_2025-10-11 15:33:41.849670_ContrastiveLoss_gru_mlp"},
           {"data": "gender", "loss": 'contrastive', 'alg':'mlp', 'path': "topo_metrics/csv_results/gender_output_2025-10-09 11:31:59.174931_ContrastiveLoss_gru_mlp"},
           {"data": "gender", "loss": 'barlow_twins', 'alg':'catboost', 'path': "topo_metrics/csv_results/gender_output_2025-10-08 23:29:01.030695_BarlowTwinsLoss_gru_catboost"},
           {"data": "gender", "loss": "vicreg", 'alg': "catboost", 'path':"topo_metrics/csv_results/gender_output_2025-10-08 10:26:11.109161_VicregLoss_gru_catboost"},
           {"data": "gender", "loss": "contrastive", 'alg': "catboost", 'path':"topo_metrics/csv_results/gender_output_2025-10-07 19:53:20.141369_ContrastiveLoss_gru_catboost"},
            {"data": "gender", "loss": "barlow_twins", 'alg': "catboost", 'path':"topo_metrics/csv_results/output_2025-10-05 22:08:41.254649 BarlowTwinsLoss default catboost downstream multimodal"},
            {"data": "gender", "loss": "contrastive", 'alg': "catboost", 'path':"topo_metrics/csv_results/output_2025-10-05 17:04:39.041865 ContrastiveLoss default catboost downstream multimodal"},
            {"data": "gender", "loss": "contrastive", 'alg': "catboost", "path":"topo_metrics/csv_results/gender_mm_contrastive_old_catboost_30"}] 

In [3]:
for i in range(len(full_exps)):
    file_paths = sorted(glob.glob(f"../{full_exps[i]['path']}/out_*.csv"))
    all_dfs = []
    #print(full_exps[i]['path'], file_paths)
    for path in file_paths:
        df = pd.read_csv(path)
        df["sample_fraction"] = float(path.split("_")[-1].replace(".csv", ""))
        all_dfs.append(df)
    
    # Объединяем в один DataFrame
    full_exps[i]['df_all'] = pd.concat(all_dfs, ignore_index=True)

In [4]:
#necessary for indexing
for i in range(len(full_exps)):
    metric_columns = [col for col in full_exps[i]['df_all'].columns if col.startswith("metric_")] 
    full_exps[i]['metric_columns'] = metric_columns
    full_exps[i]['df_all'] = full_exps[i]['df_all'].set_index('checkpoint')

In [5]:
funcs = {'Pearson': lambda metric, acc, sign: metric.corr(acc) * sign,
 'Spearman': lambda metric, acc, sign: metric.corr(acc, method='spearman') * sign,
 'Quality': lambda metric, acc, sign: acc.values[
                    np.argmax(sign * metric.values)]
}
                

In [6]:
import numpy as np
from scipy.stats import spearmanr

from collections import defaultdict

for i in range(len(full_exps)):
    # groupby type : corelation or acc : value
    corr_records = defaultdict(lambda: defaultdict(list))
    
    df_all = full_exps[i]['df_all']
    df_all['checkpoint'] = df_all.index.values
    runs = df_all['checkpoint'].values
    for j in range(df_all.shape[0]):
        runs[j] = runs[j][:-10]
    
    df_all['runs'] = runs
        
    # Correlation for each sample fraction
    frac = 1.0
    sub = df_all[df_all['sample_fraction'] == frac]
    
    #sub = sub[sub['epoch_num'] == sub['early_stop_epoch']]
    for metric in full_exps[i]['metric_columns']:
        sign = np.sign(df_all[df_all['sample_fraction'] > 0.95][metric].corr(
                sub['roc_auc'], method='spearman'))

        #OVERALL CORRELATION OVER ALL RUNS
        if sub[metric].nunique() > 1:
            for k, v in funcs.items():
                corr_records['all'][k].append({
                    "sample_fraction": frac,
                    "metric": metric,
                    "correlation": v(sub[metric], sub['roc_auc'], sign)
                })
                
        #CORRELATTION OF BEST CHECKPOINT
        for check, check_data in sub.groupby('runs'):        
            if check_data[metric].nunique() > 1 and check_data['accuracy'].nunique() > 1:
                for k, v in funcs.items():
                    corr_records['checkpoint'][k].append({
                        "sample_fraction": frac,
                        "metric": metric,
                        "correlation": v(check_data[metric], check_data['roc_auc'], sign)
                    })
                       

    for group, group_recs in corr_records.items():
        for k, curr_rec in group_recs.items():
            corr_df = pd.DataFrame(curr_rec)
            curr_name = f'corr_{k}_{group}'
            full_exps[i][f'{curr_name}_df'] = corr_df
            full_exps[i][curr_name] = {}
            full_exps[i][curr_name]['full'] = corr_df[corr_df['sample_fraction'] == 1.0].groupby("metric")["correlation"].mean().sort_values(ascending=False)
    

In [7]:

from collections import defaultdict
corr_all_full = defaultdict(lambda: defaultdict(list))
exp_keys = [k for (k, v) in full_exps[0].items() if (type(v) is str)]

corr_grouped = defaultdict(
    lambda: defaultdict(
    lambda: defaultdict(
    lambda: defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(list))))))

for i in range(len(full_exps)):
    for corr_type in funcs.keys():
        for group_type in (['all', 'checkpoint']):
            curr_name = f'corr_{corr_type}_{group_type}'
            if not(curr_name in full_exps[i]):
                continue
            for s in full_exps[i][curr_name].keys():
                corr_all_full[group_type][corr_type].append(full_exps[i][curr_name][s])
                for k in exp_keys:
                    corr_grouped[group_type][s][corr_type][k][full_exps[i][k]][full_exps[i]['data']].append(
                        full_exps[i][curr_name][s])


In [8]:
def filter_ripser(df):
    if len(df.shape) > 1:
        old_cols = df.columns
    else:
        old_cols = df.index
        
    new_cols = [col for col in old_cols 
                    if not('ripser' in col) #only one metric from a family of topologic experiments
                or 
                    col in 
                ['metric_ripser_sum_H0_norm0.9', #variant used in the article
                 ]]
    new_df = df[new_cols]
    return new_df.rename(index={col: col[7:] for col in new_df.index if 'metric' in col})

    

In [9]:
# qualiy reports, grouped by different keys
print_per_data = False
print_per_alg = False

for corr_type  in funcs.keys():
    print('================')
    print("quality type", corr_type)
    for group_type in corr_grouped.keys():
                print('*****************')
                print("grouping by", group_type)
                s = 'full'
                k = 'alg'
                
                mean_corr = []
                for k1 in corr_grouped[group_type][s][corr_type][k].keys():
                    per_data = []
                    for d_k in corr_grouped[group_type][s][corr_type][k][k1].keys():
                        curr_corr = pd.concat(corr_grouped[group_type][s][corr_type][k][k1][d_k], axis=1).mean(axis=1).sort_values(ascending=False)
                        per_data.append(curr_corr)  
                        if print_per_data:
                            print(f"quality type: {corr_type}, grouping by {group_type}, varying {k}: {k1}, data: {d_k}")
                            print(filter_ripser(curr_corr.sort_values(ascending=False)))
                    
                    curr_corr = pd.concat(per_data, axis=1).mean(axis=1) 
                    
                    if len(per_data) > 1:
                        mean_corr.append(curr_corr)
                        if print_per_alg:
                            print(f"quality type: {corr_type}, grouping by {group_type}, varying {k}: {k1}")
                            print(filter_ripser(curr_corr.sort_values(ascending=False)))
                            
                if len(mean_corr) > 1:
                    print(f"quality type: {corr_type}, grouping by {group_type}, varying {k}")
                    print(filter_ripser(
                        pd.concat(mean_corr, axis=1).mean(axis=1).sort_values(ascending=False)))

quality type Pearson
*****************
grouping by all
quality type: Pearson, grouping by all, varying alg
metric
ne_sum                     0.710347
ripser_sum_H0_norm0.9      0.671371
coherence                  0.408860
self_clustering            0.338215
rankme                     0.273637
stable_rank                0.129880
pseudo_condition_number    0.120036
alpha_req                 -0.159766
dtype: float64
*****************
grouping by checkpoint
quality type: Pearson, grouping by checkpoint, varying alg
metric
ripser_sum_H0_norm0.9      0.690681
ne_sum                     0.664789
rankme                     0.612904
self_clustering            0.597016
stable_rank                0.595298
coherence                  0.454040
pseudo_condition_number    0.246747
alpha_req                  0.115143
dtype: float64
quality type Spearman
*****************
grouping by all
quality type: Spearman, grouping by all, varying alg
metric
ripser_sum_H0_norm0.9      0.560451
ne_sum               